# Capital.com API — Kursdaten-Explorer

Ruft **alle** verfügbaren Daten zu **einem** Instrument ab. Zum Testen gedacht,
nicht Teil der Pipeline: kein Import aus `src/` oder `config.py`, nur `requests`
und `pandas`.

**Kernel:** oben rechts *Trading_Harry (venv)* wählen. Ohne den venv-Kernel fehlt `pandas`.

**Ablauf:** Zelle 1 (Konfiguration) anpassen → Zelle 2 (Login) → danach jede
Abruf-Zelle beliebig oft und in beliebiger Reihenfolge ausführen.

Die Zugangsdaten kommen aus der `.env` im Repo-Wurzelverzeichnis
(`CAPITAL_COM_API_KEY`, `CAPITAL_COM_IDENTIFIER`, `CAPITAL_COM_PASSWORD`).
Im Notebook steht kein Geheimnis.

**Fehler werden absichtlich nicht abgefangen.** Bei einem Fehlschlag erscheinen
Statuscode und Antwort-Body (dort steht Capital.coms `errorCode`), danach der
Traceback. Ein stillschweigend verschluckter Fehler wäre hier schlimmer.

## 1 · Konfiguration

Die einzige Zelle, die du normalerweise anfasst.

In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

# =============================================================================
EPIC     = "AAPL"   # Capital.com-Epic -- nicht immer gleich dem Boersen-Ticker.
                    # Beispiele: AAPL, MSFT, TSLA, GOLD, SILVER, OIL_CRUDE,
                    #            BTCUSD, ETHUSD, BRKB, SOXX, VIX
                    # Epic unbekannt? -> letzte Zelle "Epic-Suche".
USE_DEMO = True     # False = Live-API. Die .env-Daten sind Demo-Zugangsdaten.
BARS     = 200      # Bars je Aufloesung. Max. 1000 -- darueber antwortet
                    # Capital.com mit HTTP 400.
# =============================================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

_kandidaten = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _kandidaten if (p / ".env").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        f"Keine .env gefunden -- gesucht ab {Path.cwd()} aufwaerts."
    )
load_dotenv(REPO_ROOT / ".env")

_fehlend = [k for k in ("CAPITAL_COM_API_KEY", "CAPITAL_COM_PASSWORD")
            if not os.environ.get(k)]
if _fehlend:
    raise KeyError(f"In {REPO_ROOT / '.env'} fehlen: {', '.join(_fehlend)}")

API_KEY    = os.environ["CAPITAL_COM_API_KEY"]
PASSWORD   = os.environ["CAPITAL_COM_PASSWORD"]
IDENTIFIER = os.environ.get("CAPITAL_COM_IDENTIFIER") or API_KEY

BASE_URL = ("https://demo-api-capital.backend-capital.com" if USE_DEMO
            else "https://api-capital.backend-capital.com")

print(f"Epic       : {EPIC}")
print(f"Umgebung   : {'DEMO' if USE_DEMO else 'LIVE'}  ({BASE_URL})")
print(f"Bars/Aufl. : {BARS}")
print(f".env       : {REPO_ROOT / '.env'}")
print(f"API-Key    : {API_KEY[:4]}...{API_KEY[-4:]}  (Laenge {len(API_KEY)})")

## 2 · Login

`POST /api/v1/session` liefert die beiden Auth-Header `CST` und
`X-SECURITY-TOKEN`, die jeder weitere Call braucht. Beliebig oft wiederholbar —
bei einem `error.invalid.session.token` weiter unten einfach hier neu einloggen.

In [ ]:
SESSION = {}

_resp = requests.post(
    f"{BASE_URL}/api/v1/session",
    json={"identifier": IDENTIFIER, "password": PASSWORD, "encryptedPassword": False},
    headers={"X-CAP-API-KEY": API_KEY},
    timeout=30,
)
if not _resp.ok:
    print(f"POST /api/v1/session  ->  HTTP {_resp.status_code}\n{_resp.text}")
_resp.raise_for_status()

SESSION["CST"]              = _resp.headers["CST"]
SESSION["X-SECURITY-TOKEN"] = _resp.headers["X-SECURITY-TOKEN"]

print("Login ok.\n")
print(json.dumps(_resp.json(), indent=2, ensure_ascii=False))

## 3 · Helfer

Einmal ausführen. Alle folgenden Zellen bauen darauf auf.

In [ ]:
def api_get(path, **params):
    """Fuehrt einen authentifizierten GET gegen die Capital.com-API aus.

    Gibt bei einem Fehler Statuscode und Antwort-Body aus -- dort steht der
    errorCode -- und laesst danach den Traceback durch."""
    if not SESSION:
        raise RuntimeError("Nicht eingeloggt -- zuerst die Login-Zelle ausfuehren.")
    resp = requests.get(
        f"{BASE_URL}{path}",
        headers={"X-CAP-API-KEY": API_KEY, **SESSION},
        params={k: v for k, v in params.items() if v is not None},
        timeout=30,
    )
    if not resp.ok:
        print(f"GET {path} {params}  ->  HTTP {resp.status_code}\n{resp.text}")
    resp.raise_for_status()
    return resp.json()


def show_json(titel, payload):
    """Druckt eine Antwort vollstaendig und unveraendert als JSON-Block."""
    print(f"\n{'=' * 78}\n{titel}\n{'=' * 78}")
    print(json.dumps(payload, indent=2, ensure_ascii=False, default=str))


def show_table(titel, df):
    """Zeigt einen DataFrame mit Ueberschrift; meldet ausdruecklich, wenn er leer ist."""
    print(f"\n--- {titel} ({len(df)} Zeilen) ---")
    if df.empty:
        print("(leer)")
    else:
        display(df)


def flat_table(d, spalte="Feld"):
    """Klappt ein dict in einen zweispaltigen DataFrame (Feld / Wert).

    Verschachtelte Werte werden als JSON-Text abgelegt statt verworfen -- die
    Roh-Ausgabe daneben zeigt sie ohnehin vollstaendig."""
    rows = [
        {spalte: k,
         "Wert": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v}
        for k, v in d.items()
    ]
    return pd.DataFrame(rows)


def prices_to_df(prices):
    """Baut aus einer prices-Liste einen DataFrame mit bid UND ask je OHLC-Feld.

    Der Projektcode in src/providers/capital_provider.py nutzt nur die
    bid-Seite; hier bleiben beide erhalten."""
    rows = []
    for p in prices:
        row = {
            "snapshotTime":    p.get("snapshotTime"),
            "snapshotTimeUTC": p.get("snapshotTimeUTC"),
            "volume":          p.get("lastTradedVolume"),
        }
        for feld, schluessel in (("open", "openPrice"), ("high", "highPrice"),
                                 ("low", "lowPrice"), ("close", "closePrice")):
            preis = p.get(schluessel) or {}
            row[f"{feld}_bid"] = preis.get("bid")
            row[f"{feld}_ask"] = preis.get("ask")
        rows.append(row)
    return pd.DataFrame(rows)


print("Helfer bereit: api_get, show_json, show_table, flat_table, prices_to_df")

## 4 · Instrument-Details

`GET /api/v1/markets/{epic}` — der informationsreichste Call. Drei Blöcke:

| Block | Inhalt |
|---|---|
| `instrument` | Typ, Währungen, Lot-Size, Margin-Faktor, **`openingHours`**, Expiry, Land |
| `dealingRules` | min. Deal-Size, min./max. Stop- und Limit-Abstände, Order-Präferenzen |
| `snapshot` | aktueller `bid`/`offer`, Tages-High/Low, `netChange`, `marketStatus`, `updateTime` |

⚠️ `openingHours` erklärt eine Eigenheit der Tages-Bars: sie beginnen **vorbörslich**
(bei US-Aktien 08:00 UTC). Der `open` der DAY-Bar ist deshalb *nicht* der
Eröffnungskurs — dafür braucht es eine `MINUTE`-Bar.

In [ ]:
market = api_get(f"/api/v1/markets/{EPIC}")

show_json(f"GET /api/v1/markets/{EPIC}", market)

for block in ("instrument", "dealingRules", "snapshot"):
    if isinstance(market.get(block), dict):
        show_table(block, flat_table(market[block]))

## 5 · Client-Sentiment

`GET /api/v1/clientsentiment?marketIds=…` — wie viel Prozent der
Capital.com-Kunden long bzw. short in diesem Instrument sind.

⚠️ Die Doku nennt auch `/clientsentiment/item/{marketId}`. Der Pfad antwortet auf
der Demo-API **immer mit 404**, auch für Instrumente, die über die Sammelabfrage
sauber Sentiment liefern (am 2026-08-10 gegen `AAPL` und `SILVER` geprüft).
Deshalb hier bewusst die Variante mit `marketIds`.

Als `marketId` funktioniert das Epic direkt — `GET /markets/{epic}` gibt gar kein
`marketId`-Feld zurück. `marketIds` nimmt auch mehrere, komma-getrennt.

Eine leere `clientSentiments`-Liste heisst: für dieses Instrument wird kein
Sentiment geführt. Das ist kein Fehler.

In [ ]:
sentiment = api_get("/api/v1/clientsentiment", marketIds=EPIC)

show_json(f"GET /api/v1/clientsentiment?marketIds={EPIC}", sentiment)

_eintraege = sentiment.get("clientSentiments", [])
if _eintraege:
    show_table("Client-Sentiment", pd.DataFrame(_eintraege))
else:
    print(f"Kein Sentiment fuer {EPIC} verfuegbar.")

## 6 · Kursdaten — alle acht Auflösungen

`GET /api/v1/prices/{epic}` für `MINUTE`, `MINUTE_5`, `MINUTE_15`, `MINUTE_30`,
`HOUR`, `HOUR_4`, `DAY`, `WEEK`.

Je Auflösung siehst du das **erste Bar vollständig als Roh-JSON** (inklusive der
`ask`-Preise) und die komplette Tabelle mit bid *und* ask. Alle Roh-Antworten
landen in `prices_by_res`, zum Nachbohren:

```python
prices_by_res["DAY"]["prices"][0]
```

Neben `prices` liefert die Antwort drei Felder auf oberster Ebene, die sonst
leicht untergehen: `instrumentType`, `tickSize` und `pipPosition`. Sie werden je
Auflösung mit ausgegeben.

⚠️ Das sind acht Calls à `BARS` Bars. Bei knappem API-Budget `BARS` reduzieren
oder `RESOLUTIONS` kürzen.

In [ ]:
RESOLUTIONS = ["MINUTE", "MINUTE_5", "MINUTE_15", "MINUTE_30",
               "HOUR", "HOUR_4", "DAY", "WEEK"]

prices_by_res = {}

for res in RESOLUTIONS:
    antwort = api_get(f"/api/v1/prices/{EPIC}", resolution=res, max=BARS)
    prices_by_res[res] = antwort

    bars = antwort.get("prices", [])
    if bars:
        show_json(f"{res} -- erstes Bar, alle Felder ({len(bars)} Bars gesamt)", bars[0])
    else:
        print(f"\n{'=' * 78}\n{res} -- keine Bars\n{'=' * 78}")

    show_table(f"{res}", prices_to_df(bars))

    print("Antwort-Metadaten: "
          + ", ".join(f"{k}={antwort.get(k)}"
                      for k in ("instrumentType", "tickSize", "pipPosition")))

## 7 · Freies Zeitfenster

Derselbe Endpunkt mit `from`/`to` statt `max`. Format: `YYYY-MM-DDTHH:MM:SS`,
**UTC** — die API filtert auf `snapshotTimeUTC`, nicht auf lokale Zeit.

⚠️ Ein `to` in der **Zukunft** beantwortet Capital.com mit HTTP 400; fünf Minuten
genügen. Das steht nicht in der Doku, `nicht_in_der_zukunft()` klemmt es ab.

In [ ]:
RESOLUTION = "HOUR"
FROM_UTC   = (datetime.now(timezone.utc) - timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%S")
TO_UTC     = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")


def nicht_in_der_zukunft(ts):
    """Klemmt einen 'to'-Zeitstempel auf jetzt (UTC).

    Capital.com beantwortet ein 'to' in der Zukunft mit HTTP 400 -- fuenf
    Minuten genuegen bereits. Nicht dokumentiert, empirisch ermittelt."""
    jetzt = datetime.now(timezone.utc).replace(tzinfo=None)
    return min(datetime.fromisoformat(ts), jetzt).strftime("%Y-%m-%dT%H:%M:%S")


_params = {
    "resolution": RESOLUTION,
    "max":        1000,
    "from":       FROM_UTC,
    "to":         nicht_in_der_zukunft(TO_UTC),
}
print(f"Fenster: {_params}")

fenster = api_get(f"/api/v1/prices/{EPIC}", **_params)

_bars = fenster.get("prices", [])
if _bars:
    show_json(f"{RESOLUTION} -- erstes Bar ({len(_bars)} Bars im Fenster)", _bars[0])
show_table(f"{RESOLUTION}  {FROM_UTC} .. {TO_UTC}", prices_to_df(_bars))

## 8 · Session & Konto

Nicht instrumentbezogen, aber nützlich zum Gegenprüfen: welches Konto ist aktiv,
welche Währung, welcher Saldo, und in welcher Zeitzone antwortet die API.

In [ ]:
show_json("GET /api/v1/session",  api_get("/api/v1/session"))
show_json("GET /api/v1/accounts", api_get("/api/v1/accounts"))

## 9 · Epic-Suche (optional)

Gehört nicht zum Hauptablauf — Nachschlagehilfe, wenn du das Epic zu einem Namen
nicht kennst. Capital.coms Epics weichen oft vom Börsen-Ticker ab
(`OIL_CRUDE` statt `CL=F`, `BRKB` statt `BRK-B`).

Den passenden Wert aus der Spalte `epic` oben in Zelle 1 als `EPIC` eintragen.

In [ ]:
SUCHBEGRIFF = "Apple"

_treffer = api_get("/api/v1/markets", searchTerm=SUCHBEGRIFF).get("markets", [])
print(f"{len(_treffer)} Treffer fuer '{SUCHBEGRIFF}'")
show_table(f"Suche: {SUCHBEGRIFF}", pd.DataFrame(_treffer))